In [ ]:
#@title 1. Install Ollama, cloudflared and Gradio

!apt-get -qq update > /dev/null
!apt-get -qq install -y zstd pciutils psmisc > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!pip -q install gradio requests
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!nvidia-smi -L || echo "No GPU: switch to a T4 runtime"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
GPU 0: Tesla T4 (UUID: GPU-e39b57ec-81a4-d1a6-3bd9-9448aa6380c3)


In [ ]:
#@title 2. Start Ollama and pull a model

import subprocess, time, requests
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(40):
    try:
        requests.get("http://localhost:11434"); break
    except Exception:
        time.sleep(1)

MODEL = "qwen2.5:7b-instruct"
!ollama pull {MODEL}

In [ ]:
#@title 3. Run real Harness2 demo

!git clone -q https://github.com/google-research/harness2
%cd harness2
!pip -q install uv
!uv venv --python 3.12 .venv
!. .venv/bin/activate && uv pip install -e '.[render]' && harness2-demo --output runs/demo --k 2
%cd ..

fatal: destination path 'harness2' already exists and is not an empty directory.
/content/harness2
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: .venv
? A virtual environment already exists at `.venv`. Do you want to replace it? [y/n] › yes

✔ A virtual environment already exists at `.venv`. Do you want to replace it? · yes
Activate with: source .venv/bin/activate
Using Python 3.13.15 environment at: /usr
Resolved 8 packages in 66ms
Prepared 1 package in 846ms
Uninstalled 1 package in 1ms
Installed 1 package in 1ms
 ~ harness2==0.1.0 (from file:///content/harness2)
  [accounting] 1/2 skipped      ledger-1
  [accounting] 2/2 skipped      ledger-2
Local demo: 2 tasks completed; 0 failed. /content/harness2/runs/demo/demo.json
/content


In [ ]:
#@title 4. Config, anomaly taxonomy, harness

import os, re, json, time, copy, uuid, queue, threading, subprocess, hashlib, itertools, traceback, random, requests
from collections import Counter

DATA_DIR = "/content/anomaly_data"

os.makedirs(DATA_DIR, exist_ok=True)
P = lambda n: os.path.join(DATA_DIR, n)
LOG, IMP_FILE, HARNESS_FILE = P("runs_log.jsonl"), P("improvement_harness.json"), P("base_harness.json")

JUDGE_MODEL = MODEL
USE_RXCLASS = True
MAX_MEDS_PAIRWISE = 8
RX = "https://rxnav.nlm.nih.gov/REST/rxclass"



Detectors loaded.


In [ ]:
def chat(system, user, as_json=False, temperature=0.0, model=None, max_tokens=None):
    opts = {"temperature": temperature, "num_ctx": 8192}
    mt = max_tokens or (900 if as_json else None)
    if mt: opts["num_predict"] = mt
    payload = {"model": model or MODEL, "stream": False, "keep_alive": "30m", "options": opts,
               "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]}
    if as_json: payload["format"] = "json"
    for _ in range(3):
        try:
            c = requests.post("http://localhost:11434/api/chat", json=payload, timeout=900).json()["message"]["content"]
            if c.strip(): return c
        except Exception:
            pass
        time.sleep(2)
    return ""

def jload(s, default):
    try:
        v = json.loads(s)
    except Exception:
        m = re.search(r"\{.*\}", s or "", re.S)
        try: v = json.loads(m.group(0)) if m else default
        except Exception: v = default
    return v if isinstance(v, dict) else default



In [ ]:
TYPES = {
 "ALLERGY_CONFLICT": "a prescribed drug conflicts with a documented allergy",
 "DRUG_INTERACTION": "two discharge drugs interact in a clinically important way",
 "DRUG_LAB_CONDITION": "a drug is concerning given a documented lab value or condition",
 "DOSE_IMPLAUSIBLE": "a dose, unit or frequency is implausible",
 "MED_DISCREPANCY": "an unexplained difference between home and discharge medications",
 "DUPLICATE_THERAPY": "duplicate or overlapping therapy",
 "CONTRADICTION": "two statements or data points in the record disagree",
 "IMPLAUSIBLE_VALUE": "a vital sign, measurement or lab value is physiologically impossible",
 "MISSING_INFO": "information a discharge record must contain is missing",
 "DX_TX_MISMATCH": "a drug without a documented indication, or a problem without a plan",
 "TIMELINE": "dates or sequence of events are inconsistent",
}
SEV = {"high": 0, "medium": 1, "low": 2}

LLM_DETECTORS = ["contradiction", "dx_treatment", "duplicate_therapy", "sweep",
                 "dose_plausibility", "allergy", "interaction", "drug_context"]
FREE = ["contradiction", "dx_treatment", "duplicate_therapy", "sweep"]
DETECTOR_HELP = {
 "contradiction": "statements or data that disagree inside the record",
 "dx_treatment": "drugs without an indication, problems without a plan",
 "duplicate_therapy": "overlapping drugs of the same class or purpose",
 "sweep": "any other clinically important anomaly",
 "dose_plausibility": "per-drug dose and frequency plausibility",
 "allergy": "drug versus documented allergy checks",
 "interaction": "drug-drug interaction checks",
 "drug_context": "drug versus documented labs and conditions",
}

DEFAULT_HARNESS = {
 "system_prompt": ("You are a clinical safety reviewer checking a hospital discharge record for anomalies. "
                   "You work only from the record plus established pharmacology."),
 "guardrails": [
   "Report an anomaly only when the record itself supports it, and copy the supporting text verbatim into 'evidence'.",
   "Do not flag a difference or change that the record explicitly explains.",
   "Never assume facts that are not in the record.",
   "Prefer fewer, well-evidenced anomalies over many speculative ones.",
   "Laboratory tests and imaging are never medications.",
 ],
 "detector_notes": {d: [] for d in LLM_DETECTORS},
}


In [ ]:
BASE_HARNESS = json.load(open(HARNESS_FILE)) if os.path.exists(HARNESS_FILE) else copy.deepcopy(DEFAULT_HARNESS)
BASE_HARNESS.setdefault("detector_notes", {})
for _d in LLM_DETECTORS: BASE_HARNESS["detector_notes"].setdefault(_d, [])

def render_sys(h, detector=None):
    parts = [h["system_prompt"], "\nRules:"] + [f"- {g}" for g in h["guardrails"]]
    notes = h["detector_notes"].get(detector, []) if detector else []
    if notes: parts += [f"\nExtra rules for this check ({detector}):"] + [f"- {n}" for n in notes]
    return "\n".join(parts)

print("Config loaded. Model:", MODEL)

In [ ]:
#@title 5. Detectors, grounding, verification

def _norm(s): return re.sub(r"[^a-z0-9]+", " ", (s or "").lower()).strip()
def _toks(s): return set(re.findall(r"[a-z0-9]{3,}", (s or "").lower()))
def grounded(evidence, source):
    e = _norm(evidence)
    return len(e) >= 4 and e in _norm(source)

def anomaly(type_, desc, evidence, sev="medium", detector="", source="model", certain=False, pregrounded=False):
    return {"type": type_, "description": str(desc).strip()[:300], "evidence": str(evidence).strip()[:300],
            "severity": sev if sev in SEV else "medium", "detector": detector, "source": source,
            "certain": certain, "pregrounded": pregrounded}

def deidentify(t):
    """Best-effort scrub. NOT a guarantee. Review manually."""
    t = re.sub(r"(?im)^\s*(patient\s*name|name)\s*:.*$", r"\1: [REDACTED]", t)
    t = re.sub(r"(?i)\b(MRN|ID|file\s*no\.?)\s*[:#]?\s*\w+", r"\1: [REDACTED]", t)
    t = re.sub(r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b", "[DATE]", t)
    t = re.sub(r"(\+?\d[\d\s\-()]{8,}\d)", "[PHONE]", t)
    t = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", "[EMAIL]", t)
    t = re.sub(r"\b(Dr\.?|Mr\.?|Mrs\.?|Ms\.?)\s+[A-Z][a-z]+(\s+[A-Z][a-z]+)?", r"\1 [NAME]", t)
    return t

def build_source(adm, course, meds, labs):
    return (f"--- ADMISSION NOTE ---\n{adm}\n\n--- HOSPITAL COURSE / PROGRESS NOTES ---\n{course}\n\n"
            f"--- MEDICATIONS (home -> discharge) ---\n{meds}\n\n--- LABS / RESULTS / PENDING ---\n{labs}")


In [ ]:
NEG_ALLERGY = re.compile(r"\b(no|none|nil|nkda|nka|denies|not)\b", re.I)
_ex_cache = {}

def extract_structured(source):
    key = hashlib.md5(source.encode()).hexdigest()
    if key in _ex_cache: return _ex_cache[key]
    p = ("Extract structured data from this clinical record. Use ONLY what the text says. "
         "Laboratory tests, vitals and imaging are NOT medications. Use [] when something is not documented.\n\n"
         f"RECORD:\n{source}\n\n"
         'Reply JSON: {"allergies": [{"substance": "...", "example_drugs": ["up to 4 common drug ingredients in that allergy class"]}], '
         '"home_meds": [{"name": "ingredient name as written", "dose": "as written", "note": ""}], '
         '"discharge_meds": [{"name": "...", "dose": "as written", "note": "e.g. HOLD / STOPPED / NEW if written"}], '
         '"labs": ["name value unit, as written"], "conditions": ["as written"]}')
    j = jload(chat("You extract structured data from clinical text. Never invent.", p, as_json=True), {})
    low = source.lower()
    ok = lambda n: isinstance(n, str) and n.strip() and n.lower().strip() in low
    def meds(k):
        return [{"name": m["name"].strip(), "dose": str(m.get("dose", "")).strip(), "note": str(m.get("note", "")).strip()}
                for m in (j.get(k) or []) if isinstance(m, dict) and ok(m.get("name"))]
    allergies = []
    for a in j.get("allergies") or []:
        if isinstance(a, dict) and ok(a.get("substance")) and not NEG_ALLERGY.search(a["substance"]):
            allergies.append({"substance": a["substance"].strip(),
                              "example_drugs": [e for e in (a.get("example_drugs") or []) if isinstance(e, str)][:4]})
    strs = lambda k: [s for s in (j.get(k) or []) if isinstance(s, str)][:30]
    ex = {"allergies": allergies, "home": meds("home_meds"), "disch": meds("discharge_meds"),
          "labs": strs("labs"), "conditions": strs("conditions")}
    _ex_cache[key] = ex
    return ex

def dnames(ex): return list(dict.fromkeys(m["name"] for m in ex["disch"]))[:MAX_MEDS_PAIRWISE]

_rx_cache = {}
def _rx_get(path, **params):
    try:
        r = requests.get(f"{RX}/{path}.json", params=params, timeout=8)
        return r.json() if r.status_code == 200 else {}
    except Exception:
        return {}

def drug_atc_classes(name):
    key = (name or "").lower().strip()
    if key in _rx_cache: return _rx_cache[key]
    out = set()
    if USE_RXCLASS and key:
        j = _rx_get("class/byDrugName", drugName=key, relaSource="ATC")
        for it in (j.get("rxclassDrugInfoList") or {}).get("rxclassDrugInfo", []):
            cid = (it.get("rxclassMinConceptItem") or {}).get("classId", "")
            if len(cid) >= 4: out.add(cid[:4])
    _rx_cache[key] = out
    return out

def allergen_classes(al):
    cls = set()
    for nm in [al["substance"]] + al["example_drugs"]: cls |= drug_atc_classes(nm)
    return cls

def rx_selftest():
    for d in ("amoxicillin", "penicillin g", "tramadol"):
        print(d, "->", sorted(drug_atc_classes(d)) or "no classes (offline, unknown name, or parsing mismatch)")



In [ ]:
JSON_FREE = ('Reply JSON: {"anomalies": [{"type": "<one of: ' + ", ".join(TYPES) + '>", "description": "...", '
             '"evidence": "exact quote copied from the record", "severity": "high|medium|low"}]}')

def _free(name, question, needs_meds=False):
    def fn(src, ex, h):
        if needs_meds and not ex["disch"]: return []
        p = f"RECORD:\n{src}\n\nTASK: {question}\nReturn at most 5 anomalies, or an empty list if there are none. " + JSON_FREE
        j = jload(chat(render_sys(h, name), p, as_json=True), {})
        out = []
        for a in j.get("anomalies") or []:
            if isinstance(a, dict) and a.get("type") in TYPES and isinstance(a.get("description"), str) and isinstance(a.get("evidence"), str):
                out.append(anomaly(a["type"], a["description"], a["evidence"], a.get("severity", "medium"), name))
        return out
    return fn

def _ask(h, det, q, max_tokens=200):
    return jload(chat(render_sys(h, det), q, as_json=True, max_tokens=max_tokens), {})

def det_dose(src, ex, h):
    out = []
    for m in ex["disch"][:MAX_MEDS_PAIRWISE + 4]:
        if not m["dose"]: continue
        j = _ask(h, "dose_plausibility", f"Drug: {m['name']}. Dose and frequency as written: {m['dose']}. Is this implausible for an adult "
                 "(for example a 10-fold error, wrong unit, or above the usual maximum)? "
                 'Reply JSON: {"implausible": true/false, "reason": "one short sentence"}')
        if j.get("implausible") is True:
            out.append(anomaly("DOSE_IMPLAUSIBLE", f"{m['name']} {m['dose']}: {str(j.get('reason',''))[:160]}", m["name"], "high",
                               "dose_plausibility", pregrounded=True))
    return out

def det_allergy(src, ex, h):
    out = []
    for al in ex["allergies"]:
        acl = allergen_classes(al)
        for d in dnames(ex):
            shared = acl & drug_atc_classes(d)
            if shared:
                out.append(anomaly("ALLERGY_CONFLICT", f"Documented allergy '{al['substance']}' and discharge drug {d} share drug class "
                                   f"{', '.join(sorted(shared))}.", f"{al['substance']}; {d}", "high", "allergy", "rxclass", pregrounded=True))
                continue
            j = _ask(h, "allergy", f"Patient allergy: {al['substance']}. Discharge drug: {d}. Is this drug in the same class as the allergen, "
                     "or known to have clinically relevant cross-reactivity with it? Be conservative. "
                     'Reply JSON: {"conflict": true/false, "reason": "one short sentence"}')
            if j.get("conflict") is True:
                out.append(anomaly("ALLERGY_CONFLICT", f"Allergy '{al['substance']}' vs discharge drug {d}: {str(j.get('reason',''))[:160]}",
                                   f"{al['substance']}; {d}", "high", "allergy", pregrounded=True))
    return out

def det_interaction(src, ex, h):
    out = []
    for a, b in itertools.combinations(dnames(ex), 2):
        j = _ask(h, "interaction", f"Drug A: {a}. Drug B: {b}. Is there a well-established MAJOR or MODERATE interaction when taken together? "
                 'Reply JSON: {"interaction": true/false, "severity": "major|moderate|minor|none", "reason": "one short sentence"}')
        if j.get("interaction") is True and j.get("severity") in ("major", "moderate"):
            out.append(anomaly("DRUG_INTERACTION", f"{a} + {b} ({j['severity']}): {str(j.get('reason',''))[:160]}", f"{a}; {b}",
                               "high" if j["severity"] == "major" else "medium", "interaction", pregrounded=True))
    return out

def det_context(src, ex, h):
    ctx = "; ".join(ex["labs"] + ex["conditions"])
    out = []
    if not ctx: return out
    for d in dnames(ex):
        j = _ask(h, "drug_context", f"Discharge drug: {d}. Documented labs/conditions: {ctx}. Using ONLY the documented values, is there a value or "
                 "condition that makes this drug concerning (contraindication, dose adjustment, monitoring need)? "
                 'Reply JSON: {"concern": true/false, "reason": "name the specific value/condition and why"}')
        if j.get("concern") is True:
            out.append(anomaly("DRUG_LAB_CONDITION", f"{d}: {str(j.get('reason',''))[:200]}", d, "high", "drug_context", pregrounded=True))
    return out



In [ ]:
DET = {
 "contradiction": _free("contradiction", "Find statements in the record that contradict other statements or data in the same record "
                        "(for example the narrative says afebrile but a recorded temperature is high, or dates or counts that disagree)."),
 "dx_treatment": _free("dx_treatment", "Find mismatches between diagnoses/problems and treatment: a medication with no documented indication, "
                       "or a documented problem with no plan or treatment."),
 "duplicate_therapy": _free("duplicate_therapy", "Find duplicate or overlapping therapy among the discharge medications that is unsafe "
                            "(for example two drugs of the same class, or an ACE inhibitor with an ARB).", needs_meds=True),
 "sweep": _free("sweep", "Find any other clinically important anomaly a careful reviewer would flag that is not simply a missing optional detail."),
 "dose_plausibility": det_dose, "allergy": det_allergy, "interaction": det_interaction, "drug_context": det_context,
}

VALUE_RULES = [
 (r"(\d+(?:\.\d+)?)\s?kg\b", 2, 300, "weight"),
 (r"(\d+(?:\.\d+)?)\s?cm\b", 30, 250, "height"),
 (r"\btemp(?:erature)?\s*[:=]?\s*(\d+(?:\.\d+)?)\s?°?\s?c\b", 30, 43, "temperature"),
 (r"\b(?:spo2|o2 sat\w*|oxygen sat\w*)\s*[:=]?\s*(\d+(?:\.\d+)?)", 50, 100, "oxygen saturation"),
 (r"\b(?:hr|heart rate|pulse)\s*[:=]?\s*(\d+)", 20, 250, "heart rate"),
 (r"\b(?:rr|resp\w* rate)\s*[:=]?\s*(\d+)", 4, 70, "respiratory rate"),
]

def _key(n):
    n = (n or "").strip()
    return re.sub(r"[^a-z]", "", n.lower().split()[0]) if n else ""

def med_diff(home, disch):
    H = {_key(m["name"]): m for m in home}; D = {_key(m["name"]): m for m in disch}; out = []
    for k, m in D.items():
        if home and k not in H:
            out.append((f"{m['name']} is on the discharge list but not the home list.", m["name"]))
        elif k in H and H[k]["dose"] and m["dose"] and H[k]["dose"].lower() != m["dose"].lower():
            out.append((f"{m['name']}: dose differs between home ({H[k]['dose']}) and discharge ({m['dose']}).", m["name"]))
        if re.search(r"hold|stop|discontinu|withh", m.get("note", "").lower()):
            out.append((f"{m['name']} is marked '{m['note']}' on the discharge list.", m["name"]))
    for k, m in H.items():
        if disch and k not in D:
            out.append((f"{m['name']} is on the home list but missing from the discharge list.", m["name"]))
    return out

def code_detectors(src, ex):
    low, out = src.lower(), []
    for pat, lo, hi, nm in VALUE_RULES:
        for m in re.finditer(pat, low):
            v = float(m.group(1))
            if v < lo or v > hi:
                out.append(anomaly("IMPLAUSIBLE_VALUE", f"Implausible {nm}: {m.group(0).strip()}.", m.group(0).strip(), "high", "code", "code", True, True))
    if not re.search(r"allerg|nkda|\bnka\b", low):
        out.append(anomaly("MISSING_INFO", "No allergy status is documented.", "(allergy status absent)", "high", "code", "code", True, True))
    if not re.search(r"follow[- ]?up|\bclinic\b|appointment|review in|\bgp\b", low):
        out.append(anomaly("MISSING_INFO", "No follow-up plan is documented.", "(follow-up absent)", "medium", "code", "code", True, True))
    if not re.search(r"condition at discharge|\bstable\b|improved|discharged (?:home|in)|afebrile", low):
        out.append(anomaly("MISSING_INFO", "Condition at discharge is not documented.", "(discharge condition absent)", "medium", "code", "code", False, True))
    if not ex["home"] and not ex["disch"]:
        has_dose = bool(re.search(r"\d\s?(?:mg|mcg|µg|g|iu|units?)\b", low))
        out.append(anomaly("MISSING_INFO", "No medication list is documented.", "(medication list absent)", "high", "code", "code", not has_dose, True))
    a = re.search(r"admi\w*(?: date)?\s*[:\-]?\s*(\d{4}-\d{2}-\d{2})", low)
    d = re.search(r"discharge\w*(?: date)?\s*[:\-]?\s*(\d{4}-\d{2}-\d{2})", low)
    if a and d and d.group(1) < a.group(1):
        out.append(anomaly("TIMELINE", f"Discharge date {d.group(1)} is before admission date {a.group(1)}.", f"admission {a.group(1)}; discharge {d.group(1)}",
                           "high", "code", "code", True, True))
    for desc, name in med_diff(ex["home"], ex["disch"]):
        out.append(anomaly("MED_DISCREPANCY", desc, name, "medium", "code", "code", False, True))
    return out

In [ ]:
def same(a, b):
    if a["type"] != b["type"]: return False
    t, u = _toks(a["description"] + " " + a["evidence"]), _toks(b["description"] + " " + b["evidence"])
    return len(t & u) / max(len(t | u), 1) > 0.4

def dedupe(items):
    keep = []
    for a in items:
        dup = next((x for x in keep if same(a, x)), None)
        if dup:
            if SEV[a["severity"]] < SEV[dup["severity"]]: dup["severity"] = a["severity"]
            dup["certain"] = dup["certain"] or a["certain"]
        else:
            keep.append(dict(a))
    return keep

_ver_cache = {}
def verify(src, a):
    key = hashlib.md5((src + a["type"] + a["description"] + a["evidence"]).encode()).hexdigest()
    if key in _ver_cache: return _ver_cache[key]
    p = (f"RECORD:\n{src}\n\nCANDIDATE ANOMALY\nType: {a['type']}\nDescription: {a['description']}\nEvidence: {a['evidence']}\n\n"
         "Is this a genuine problem that a clinician reviewing this discharge would want flagged? Answer false if the record explains it, "
         "if it is speculative, or if it is normal and expected. "
         'Reply JSON: {"genuine": true/false, "reason": "one short sentence"}')
    j = jload(chat("You are a skeptical senior clinician verifying safety flags.", p, as_json=True, model=JUDGE_MODEL, max_tokens=150), {})
    res = (j.get("genuine") is True, str(j.get("reason", ""))[:200])
    _ver_cache[key] = res
    return res

def finalize(src, cands):
    ung, good = [], []
    for a in cands:
        (good if (a["pregrounded"] or grounded(a["evidence"], src)) else ung).append(a)
    verified, rejected = [], []
    for a in dedupe(good):
        if a["certain"]: verified.append(a); continue
        ok, why = verify(src, a)
        (verified if ok else rejected).append(dict(a, reason=why))
    return {"verified": verified, "rejected": rejected, "ungrounded": ung}

print("Detectors loaded.")

In [ ]:
#@title 6. Recursive refinement: Propose–Probe–Compose + improvement harness

def load_imp():
    if os.path.exists(IMP_FILE): return json.load(open(IMP_FILE))
    return {"editor_guidance": ["Propose short, general, reusable rules that make detectors more precise (verbatim evidence, skip explained differences) "
                                "or more complete (systematically check one specific category)."],
            "lessons": []}
def save_imp(i): json.dump(i, open(IMP_FILE, "w"), indent=1)

def eligible_lessons(imp):
    return [l for l in imp["lessons"] if not l["gains"] or sum(l["gains"]) / len(l["gains"]) > 0][-8:]

def add_lesson(text, task="anomaly"):
    imp = load_imp(); t = _toks(text)
    for x in imp["lessons"]:
        u = _toks(x["text"])
        if len(t & u) / max(len(t | u), 1) > 0.45: return x["id"]
    new = {"id": uuid.uuid4().hex[:8], "text": text, "gains": []}
    imp["lessons"] = (imp["lessons"] + [new])[-40:]; save_imp(imp)
    return new["id"]

def credit_lessons(ids, gain):
    """Called only from the benchmark, where the true anomalies are known."""
    imp = load_imp()
    for l in imp["lessons"]:
        if l["id"] in ids: l["gains"].append(gain)
    imp["lessons"] = [l for l in imp["lessons"]
                      if not (len(l["gains"]) >= 2 and sum(l["gains"]) / len(l["gains"]) <= 0) and not (l["gains"] and min(l["gains"]) < -30)]
    save_imp(imp)


In [ ]:

BAD_EDIT = re.compile(r"assum|infer|guess|always flag|flag every|flag all|if not (specified|documented|provided)|typical|default to", re.I)

def brief(items, n=6):
    return "\n".join(f"- [{a['type']}] {a['description']}" for a in items[:n]) or "(none)"

def propose_edits(cur, h, imp, used):
    les = eligible_lessons(imp); used.update(l["id"] for l in les)
    rej = Counter(a["detector"] for a in cur["rejected"] + cur["ungrounded"])
    ver = Counter(a["detector"] for a in cur["verified"])
    target = max(LLM_DETECTORS, key=lambda d: (rej[d], -ver[d]))
    p = ("\n".join(imp["editor_guidance"]) + "\n\nLessons from earlier benchmark runs:\n" +
         ("\n".join(f"- {l['text']}" for l in les) or "(none yet)") +
         f"\n\nVerified anomalies in the current run:\n{brief(cur['verified'])}\n\n"
         f"Candidates rejected by verification (likely false positives):\n{brief(cur['rejected'])}\n\n"
         f"Candidates dropped because their evidence was not in the record: {len(cur['ungrounded'])}\n\n"
         f"Propose exactly two NEW rules: (1) a GENERAL rule for all detectors; (2) a rule for the '{target}' detector ({DETECTOR_HELP[target]}). "
         "Each rule is one short sentence that makes detection more precise or more complete. Never tell a detector to assume, infer or guess. "
         "Do not mention this patient. " 'Reply JSON: {"general": "...", "specific": "..."}')
    j = jload(chat("You improve the rules of clinical anomaly detectors.", p, as_json=True, temperature=0.7, max_tokens=300), {})
    out = []
    for scope, key in (("general", "general"), (target, "specific")):
        e = j.get(key)
        if isinstance(e, str):
            e = e.strip()
            if 10 <= len(e) <= 300 and not BAD_EDIT.search(e) and e not in h["guardrails"] + h["detector_notes"].get(scope, []):
                out.append({"scope": scope, "edit": e})
    return out

def apply_edit(h, e):
    h2 = copy.deepcopy(h)
    if e["scope"] == "general": h2["guardrails"].append(e["edit"])
    else: h2["detector_notes"].setdefault(e["scope"], []).append(e["edit"])
    return h2

flat = lambda raw: [a for n in raw for a in raw[n]]

def contrast(base, probe):
    gained = [a for a in probe["verified"] if not any(same(a, b) for b in base["verified"])]
    lost = [b for b in base["verified"] if not any(same(a, b) for a in probe["verified"])]
    bad_b, bad_p = base["rejected"] + base["ungrounded"], probe["rejected"] + probe["ungrounded"]
    new_bad = [a for a in bad_p if not any(same(a, b) for b in bad_b)]
    return gained, lost, new_bad

def detect(src, k=1, cb=None):
    cb = cb or (lambda m: None)
    imp = load_imp(); used = set()
    cb("Extracting structure")
    ex = extract_structured(src)
    h = copy.deepcopy(BASE_HARNESS)
    raw = {"code": code_detectors(src, ex)}
    for n in LLM_DETECTORS:
        cb(f"Baseline detector: {n}"); raw[n] = DET[n](src, ex, h)
    cb("Grounding and verifying candidates")
    base = finalize(src, flat(raw))
    cur, adopted, touched, general_adopted = base, [], set(), False

    for s in range(int(k)):
        cb(f"Step {s+1}/{k}: proposing rules")
        edits = propose_edits(cur, h, imp, used)
        step = []
        for e in edits:
            names = FREE if e["scope"] == "general" else [e["scope"]]
            h2 = apply_edit(h, e); raw2 = dict(raw)
            for n in names:
                cb(f"  probing '{e['scope']}' rule on detector: {n}"); raw2[n] = DET[n](src, ex, h2)
            g, l, nb = contrast(cur, finalize(src, flat(raw2)))
            ok = len(g) >= 1 and not any(a["severity"] == "high" for a in l) and len(nb) <= len(g)
            cb(f"  rule for '{e['scope']}': +{len(g)} new, -{len(l)} lost, +{len(nb)} new rejects -> {'ADOPT' if ok else 'skip'}")
            if ok: step.append((e, names))
        if step:
            rerun = set()
            for e, names in step:
                h = apply_edit(h, e); adopted.append(e); rerun |= set(names)
                general_adopted = general_adopted or e["scope"] == "general"
            touched |= rerun
            for n in rerun:
                cb(f"  carrying refined harness forward: {n}"); raw[n] = DET[n](src, ex, h)
            cur = finalize(src, flat(raw))

    if general_adopted:
        for n in [d for d in LLM_DETECTORS if d not in touched]:
            cb(f"Final pass with refined harness: {n}"); raw[n] = DET[n](src, ex, h)
        cur = finalize(src, flat(raw))
    final = cur if adopted else base

    lesson_ids = set(used)
    if adopted:
        cb("Reflecting: updating the improvement harness")
        rp = ("These detector rules were adopted because they surfaced additional verified anomalies without losing high-severity ones:\n" +
              "\n".join(f"- ({e['scope']}) {e['edit']}" for e in adopted) +
              '\nWrite ONE short general lesson (max 25 words) about what kind of rule helps. Reply JSON: {"lesson": "..."}')
        t = jload(chat("You write concise lessons.", rp, as_json=True, max_tokens=120), {}).get("lesson")
        if isinstance(t, str) and t.strip(): lesson_ids.add(add_lesson(t.strip()))
    return {"baseline": base, "final": final, "adopted": adopted, "lesson_ids": list(lesson_ids), "ex": ex}

def run_record(src, k=1, cb=None):
    t0 = time.time(); r = detect(src, k, cb)
    rec = {"kind": "run", "ts": time.strftime("%Y-%m-%d %H:%M:%S"), "k": int(k), "secs": round(time.time() - t0),
           "n_base": len(r["baseline"]["verified"]), "n_final": len(r["final"]["verified"]),
           "n_rejected": len(r["final"]["rejected"]), "n_ungrounded": len(r["final"]["ungrounded"]),
           "adopted": len(r["adopted"]), "types": dict(Counter(a["type"] for a in r["final"]["verified"]))}
    open(LOG, "a").write(json.dumps(rec) + "\n")   # counts only, no record text
    return r, rec
print("Refinement loaded.")

In [ ]:
#@title 7. Seeded-anomaly benchmark, scoring, insights, promotion

TEMPLATES = {
 "pneumonia": dict(pt="67F", adm_date="2026-03-01", dis_date="2026-03-05",
   diagnoses="Community-acquired pneumonia (right lower lobe); hypertension; hyperlipidemia",
   presentation="4 days of productive cough, fever and dyspnea. SpO2 88% on room air on arrival, chest X-ray right lower lobe consolidation.",
   allergies="No known drug allergies", pmh="Hypertension, hyperlipidemia",
   course="Treated with IV ceftriaxone and azithromycin, then switched to oral antibiotics. Oxygen weaned to room air by day 3. Afebrile since day 3 and improved.",
   vitals="Temp 36.8 C, HR 78 bpm, RR 16, SpO2 97%, BP 128/76 mmHg, Weight 74 kg",
   labs=["WBC 16.2 -> 9.1 x10^9/L", "Creatinine 0.9 mg/dL", "Potassium 4.1 mmol/L"],
   home=[("amlodipine", "5 mg daily", ""), ("atorvastatin", "20 mg nightly", "")],
   disch=[("amlodipine", "5 mg daily", ""), ("atorvastatin", "20 mg nightly", ""), ("amoxicillin-clavulanate", "875/125 mg BD for 3 more days", "NEW, to complete antibiotic course")],
   pending="Blood cultures x2 final result pending", followup="GP review in 1 week; repeat chest X-ray in 6 weeks",
   condition="Stable, afebrile, ambulating independently"),
 "heart_failure": dict(pt="72M", adm_date="2026-04-10", dis_date="2026-04-16",
   diagnoses="Acute decompensated heart failure (HFrEF, EF 30%); atrial fibrillation; hypertension",
   presentation="5 days of worsening dyspnea, orthopnea and leg edema.",
   allergies="No known drug allergies", pmh="HFrEF, hypertension, atrial fibrillation",
   course="Diuresed with IV furosemide, net negative 4.5 L, discharge weight 82 kg. Echocardiogram performed. Renal function stable. Euvolemic and afebrile at discharge.",
   vitals="Temp 36.6 C, HR 70 bpm, RR 16, SpO2 96%, BP 118/72 mmHg, Weight 82 kg",
   labs=["Creatinine 1.1 mg/dL", "Potassium 4.3 mmol/L"],
   home=[("furosemide", "40 mg daily", ""), ("lisinopril", "20 mg daily", ""), ("apixaban", "5 mg BD", ""), ("carvedilol", "12.5 mg BD", "")],
   disch=[("furosemide", "80 mg daily", "dose increased for fluid overload"), ("lisinopril", "20 mg daily", ""), ("apixaban", "5 mg BD", ""), ("carvedilol", "12.5 mg BD", "")],
   pending="Formal echocardiogram report pending", followup="Heart failure clinic in 2 weeks; basic metabolic panel in 1 week",
   condition="Stable, euvolemic, ambulating"),
 "cholecystectomy": dict(pt="45F", adm_date="2026-05-02", dis_date="2026-05-03",
   diagnoses="Symptomatic cholelithiasis; depression",
   presentation="Recurrent biliary colic. Laparoscopic cholecystectomy performed.",
   allergies="No known drug allergies", pmh="Depression",
   course="Uncomplicated laparoscopic cholecystectomy. Tolerating diet, pain controlled, port sites clean. Afebrile and discharged on postoperative day 1.",
   vitals="Temp 36.9 C, HR 74 bpm, RR 14, SpO2 98%, BP 120/74 mmHg, Weight 68 kg",
   labs=["Hemoglobin 12.8 g/dL", "Creatinine 0.8 mg/dL"],
   home=[("sertraline", "50 mg daily", ""), ("ibuprofen", "400 mg PRN", "")],
   disch=[("sertraline", "50 mg daily", ""), ("acetaminophen", "1 g q6h PRN", ""), ("ibuprofen", "400 mg PRN", "")],
   pending="Gallbladder pathology pending", followup="Surgical clinic in 2 weeks",
   condition="Stable, tolerating diet, pain controlled"),
}

def render_record(r):
    L = [f"Admission date: {r['adm_date']}", f"Discharge date: {r['dis_date']}", f"Patient: {r['pt']}",
         f"Diagnoses: {r['diagnoses']}", f"Presentation: {r['presentation']}"]
    if r["allergies"]: L.append(f"Allergies: {r['allergies']}")
    L += [f"Past history: {r['pmh']}", f"Hospital course: {r['course']}", f"Vitals at discharge: {r['vitals']}",
          "Labs: " + "; ".join(r["labs"]),
          "Home medications: " + "; ".join(f"{n} {d}".strip() for n, d, _ in r["home"]),
          "Discharge medications: " + "; ".join(f"{n} {d}" + (f" ({note})" if note else "") for n, d, note in r["disch"])]
    if r["pending"]: L.append(f"Pending results: {r['pending']}")
    if r["followup"]: L.append(f"Follow-up: {r['followup']}")
    L.append(f"Condition at discharge: {r['condition']}")
    return "\n".join(L)

def _has(r, name): return any(n == name for n, _, _ in r["disch"])

def _m_allergy(r):
    for med, al, keys in [("amoxicillin-clavulanate", "penicillin (rash)", ["amoxicillin", "penicillin"]),
                          ("lisinopril", "ACE inhibitors (angioedema)", ["lisinopril", "angioedema", "ace inhibitor"]),
                          ("ibuprofen", "NSAIDs (urticaria)", ["ibuprofen", "nsaid"])]:
        if _has(r, med):
            r2 = copy.deepcopy(r); r2["allergies"] = al; return r2, {"type": "ALLERGY_CONFLICT", "keys": keys}

def _m_interaction(r):
    for have, add, keys in [("sertraline", ("tramadol", "50 mg q6h PRN", "NEW for pain"), ["tramadol", "sertraline"]),
                            ("apixaban", ("ibuprofen", "400 mg TID", "NEW for pain"), ["ibuprofen", "apixaban"]),
                            ("atorvastatin", ("clarithromycin", "500 mg BD", "NEW"), ["clarithromycin", "atorvastatin"])]:
        if _has(r, have):
            r2 = copy.deepcopy(r); r2["disch"].append(add); return r2, {"type": "DRUG_INTERACTION", "keys": keys}

def _m_duplicate(r):
    for have, add, keys in [("ibuprofen", ("naproxen", "500 mg BD", "NEW"), ["naproxen", "ibuprofen"]),
                            ("lisinopril", ("losartan", "50 mg daily", "NEW"), ["losartan", "lisinopril"]),
                            ("amlodipine", ("nifedipine", "30 mg daily", "NEW"), ["nifedipine", "amlodipine"])]:
        if _has(r, have):
            r2 = copy.deepcopy(r); r2["disch"].append(add); return r2, {"type": "DUPLICATE_THERAPY", "keys": keys}

def _m_dose(r):
    for i in range(len(r["disch"]) - 1, -1, -1):
        n, d, note = r["disch"][i]; m = re.match(r"^(\d+) mg\b(.*)", d)
        if m:
            r2 = copy.deepcopy(r); r2["disch"][i] = (n, f"{int(m.group(1)) * 10} mg{m.group(2)}", note)
            return r2, {"type": "DOSE_IMPLAUSIBLE", "keys": [n]}

def _m_value(r):
    r2 = copy.deepcopy(r); r2["vitals"] = re.sub(r"Weight [\d.]+ kg", "Weight 999 kg", r2["vitals"])
    return r2, {"type": "IMPLAUSIBLE_VALUE", "keys": ["999", "weight"]}

def _m_contradiction(r):
    r2 = copy.deepcopy(r); r2["vitals"] = re.sub(r"Temp [\d.]+ C", "Temp 38.9 C", r2["vitals"])
    return r2, {"type": "CONTRADICTION", "keys": ["afebrile", "38.9", "temp", "fever"]}

def _m_no_allergy(r):
    r2 = copy.deepcopy(r); r2["allergies"] = ""; return r2, {"type": "MISSING_INFO", "keys": ["allerg"]}

def _m_no_followup(r):
    r2 = copy.deepcopy(r); r2["followup"] = ""; return r2, {"type": "MISSING_INFO", "keys": ["follow", "clinic", "appointment", "pending"]}

def _m_timeline(r):
    r2 = copy.deepcopy(r); r2["adm_date"], r2["dis_date"] = r["dis_date"], r["adm_date"]
    return r2, {"type": "TIMELINE", "keys": ["date", "admission", "discharge"]}

def _m_unexplained(r):
    for n, _, _ in reversed(r["home"]):
        if _has(r, n):
            r2 = copy.deepcopy(r); r2["disch"] = [m for m in r2["disch"] if m[0] != n]
            return r2, {"type": "MED_DISCREPANCY", "keys": [n]}

def _m_dx_tx(r):
    r2 = copy.deepcopy(r); r2["disch"].append(("levothyroxine", "100 mcg daily", "NEW"))
    return r2, {"type": "DX_TX_MISMATCH", "keys": ["levothyroxine"]}

def _m_lab(r):
    r2 = copy.deepcopy(r)
    r2["home"].append(("metformin", "1000 mg BD", "")); r2["disch"].append(("metformin", "1000 mg BD", ""))
    r2["labs"] = [l for l in r2["labs"] if "creatinine" not in l.lower()] + ["Creatinine 3.4 mg/dL, eGFR 18 mL/min"]
    return r2, {"type": "DRUG_LAB_CONDITION", "keys": ["metformin", "creatinine", "egfr"]}


In [ ]:

MUTATORS = {"allergy_conflict": _m_allergy, "interaction": _m_interaction, "duplicate": _m_duplicate, "dose_error": _m_dose,
            "implausible_value": _m_value, "contradiction": _m_contradiction, "missing_allergy": _m_no_allergy,
            "missing_followup": _m_no_followup, "timeline": _m_timeline, "unexplained_change": _m_unexplained,
            "dx_tx_mismatch": _m_dx_tx, "drug_lab": _m_lab}

def all_cases():
    cases = []
    for tn, t in TEMPLATES.items():
        cases.append((f"{tn}/clean", render_record(t), []))
        for mn, fn in MUTATORS.items():
            res = fn(t)
            if res: cases.append((f"{tn}/{mn}", render_record(res[0]), [res[1]]))
    return cases

def pick_cases(n, seed):
    cases = all_cases(); rng = random.Random(seed)
    clean = [c for c in cases if not c[2]]; rest = [c for c in cases if c[2]]
    picked = clean[:min(len(clean), max(1, n // 4))]
    picked += rng.sample(rest, min(len(rest), max(0, n - len(picked))))
    return picked

COMPAT = {
 "ALLERGY_CONFLICT": {"ALLERGY_CONFLICT"}, "DRUG_INTERACTION": {"DRUG_INTERACTION", "DUPLICATE_THERAPY"},
 "DUPLICATE_THERAPY": {"DUPLICATE_THERAPY", "DRUG_INTERACTION"}, "DOSE_IMPLAUSIBLE": {"DOSE_IMPLAUSIBLE", "IMPLAUSIBLE_VALUE"},
 "IMPLAUSIBLE_VALUE": {"IMPLAUSIBLE_VALUE", "CONTRADICTION"}, "CONTRADICTION": {"CONTRADICTION"}, "MISSING_INFO": {"MISSING_INFO"},
 "MED_DISCREPANCY": {"MED_DISCREPANCY", "DX_TX_MISMATCH"}, "DX_TX_MISMATCH": {"DX_TX_MISMATCH", "MED_DISCREPANCY"},
 "DRUG_LAB_CONDITION": {"DRUG_LAB_CONDITION", "DOSE_IMPLAUSIBLE"}, "TIMELINE": {"TIMELINE"},
}


In [ ]:
def matches(a, label):
    txt = (a["description"] + " " + a["evidence"]).lower()
    return a["type"] in COMPAT[label["type"]] and any(k in txt for k in label["keys"])

def score_case(found, labels):
    hits = [[l["type"], any(matches(a, l) for a in found)] for l in labels]
    good = sum(1 for a in found if any(matches(a, l) for l in labels))
    tp = sum(1 for _, h in hits if h)
    if not labels: f1 = 100.0 if not found else 0.0
    else:
        rec = tp / len(labels); prec = (good / len(found)) if found else 0.0
        f1 = 0.0 if rec + prec == 0 else 200 * prec * rec / (prec + rec)
    return {"f1": round(f1, 1), "tp": tp, "n_labels": len(labels), "n_found": len(found), "fp": len(found) - good, "hits": hits}

def benchmark(n=8, k=1, seed=0, cb=print):
    """k=0 measures the baseline harness only. k>=1 also measures refinement and credits lessons."""
    cases = pick_cases(int(n), int(seed)); rows = []
    for i, (cid, text, labels) in enumerate(cases, 1):
        cb(f"[{i}/{len(cases)}] {cid}"); t0 = time.time()
        r = detect(text, int(k))
        sb, sf = score_case(r["baseline"]["verified"], labels), score_case(r["final"]["verified"], labels)
        gain = round(sf["f1"] - sb["f1"], 1)
        if r["adopted"]: credit_lessons(r["lesson_ids"], gain)
        rej = Counter(a["detector"] for a in r["final"]["rejected"] + r["final"]["ungrounded"])
        rec = {"kind": "bench", "ts": time.strftime("%Y-%m-%d %H:%M:%S"), "case": cid, "k": int(k), "label_types": [l["type"] for l in labels],
               "base": sb, "final": sf, "hits_base": sb["hits"], "hits_final": sf["hits"], "gain": gain,
               "adopted": [{"scope": e["scope"], "edit": e["edit"]} for e in r["adopted"]], "rejected_by_det": dict(rej), "secs": round(time.time() - t0)}
        open(LOG, "a").write(json.dumps(rec) + "\n"); rows.append(rec)
        missed = [t for t, h in sb["hits"] if not h]
        cb(f"    F1 {sb['f1']} -> {sf['f1']} | found {sb['n_found']} -> {sf['n_found']} | false positives {sb['fp']} -> {sf['fp']}"
           + (f" | missed (baseline): {missed}" if missed else "") + f" | adopted {len(r['adopted'])} | {rec['secs']}s")
    avg = lambda xs: round(sum(xs) / len(xs), 1) if xs else 0
    cb(f"\nMEAN F1: baseline {avg([r['base']['f1'] for r in rows])} -> refined {avg([r['final']['f1'] for r in rows])}")
    return rows

def insights_report():
    if not os.path.exists(LOG): return "No runs yet."
    rows = [json.loads(l) for l in open(LOG)]
    bench = [r for r in rows if r.get("kind") == "bench"]; runs = [r for r in rows if r.get("kind") == "run"]
    avg = lambda xs: round(sum(xs) / len(xs), 1) if xs else 0
    o = [f"## Insights: {len(bench)} benchmark case(s), {len(runs)} live run(s)\n"]
    if bench:
        o.append(f"**Benchmark F1 (0-100):** baseline {avg([r['base']['f1'] for r in bench])} -> refined {avg([r['final']['f1'] for r in bench])} "
                 f"(mean gain {avg([r['gain'] for r in bench]):+})")
        by = {}
        for r in bench:
            for (t, hb), (_, hf) in zip(r["hits_base"], r["hits_final"]):
                x = by.setdefault(t, [0, 0, 0]); x[0] += 1; x[1] += int(hb); x[2] += int(hf)
        o.append("\n**Recall by anomaly type (baseline -> refined):**\n" +
                 "\n".join(f"- {t}: {b}/{n} -> {f}/{n}" for t, (n, b, f) in sorted(by.items())))
        clean = [r for r in bench if not r["label_types"]]
        if clean:
            o.append(f"\n**False positives on clean records (should be 0):** baseline {avg([r['base']['n_found'] for r in clean])} -> refined {avg([r['final']['n_found'] for r in clean])} per record ({len(clean)} clean cases)")
        half = max(len(bench) // 2, 1)
        if len(bench) > 1: o.append(f"\n**Gain trend (first half -> second half):** {avg([r['gain'] for r in bench[:half]]):+} -> {avg([r['gain'] for r in bench[half:]]):+}")
        rj = Counter()
        for r in bench: rj.update(r.get("rejected_by_det", {}))
        o.append("\n**Detectors producing the most rejected candidates (false-positive sources):** " + (", ".join(f"{d} x{c}" for d, c in rj.most_common(4)) or "none"))
        st = {}
        for r in bench:
            for e in r["adopted"]: st.setdefault((e["scope"], e["edit"]), []).append(r["gain"])
        o.append("\n**Adopted rules (times, avg gain):**\n" + ("\n".join(f"- [{s}] {e} (x{len(g)}, {avg(g):+})" for (s, e), g in sorted(st.items(), key=lambda kv: -len(kv[1]))[:6]) or "- none"))
    if runs: o.append(f"\n**Live runs:** avg anomalies {avg([r['n_final'] for r in runs])}, avg rejected {avg([r['n_rejected'] for r in runs])}")
    les = [l for l in load_imp()["lessons"] if l["gains"]]; les.sort(key=lambda l: -sum(l["gains"]) / len(l["gains"]))
    o.append("\n**Lessons ranked by benchmark gain:**\n" + ("\n".join(f"- [{sum(l['gains'])/len(l['gains']):+.1f}, n={len(l['gains'])}] {l['text']}" for l in les[:6]) or "- none yet"))
    return "\n".join(o)

def promote(min_count=2):
    """Bake rules adopted at least min_count times with positive average benchmark gain into the base harness."""
    rows = [json.loads(l) for l in open(LOG)] if os.path.exists(LOG) else []
    st = {}
    for r in rows:
        if r.get("kind") == "bench":
            for e in r["adopted"]: st.setdefault((e["scope"], e["edit"]), []).append(r["gain"])
    new = []
    for (scope, edit), g in st.items():
        if len(g) >= min_count and sum(g) / len(g) > 0:
            tgt = BASE_HARNESS["guardrails"] if scope == "general" else BASE_HARNESS["detector_notes"].setdefault(scope, [])
            if edit not in tgt: tgt.append(edit); new.append((scope, edit))
    json.dump(BASE_HARNESS, open(HARNESS_FILE, "w"), indent=2)
    return new
print("Benchmark loaded. Cases available:", len(all_cases()))

In [ ]:
#@title 8. Smoke test

rx_selftest()

case = [c for c in all_cases() if c[0] == "pneumonia/allergy_conflict"][0]
r, rec = run_record(case[1], k=0, cb=print)
for a in sorted(r["final"]["verified"], key=lambda a: SEV[a["severity"]]):
    print(f"[{a['severity']}] {a['type']}: {a['description']}  (evidence: {a['evidence']})")
print("rejected:", len(r["final"]["rejected"]), "| ungrounded:", len(r["final"]["ungrounded"]))


benchmark(n=8, k=0, seed=0)
# benchmark(n=8, k=1, seed=0)

In [ ]:
#@title 9. App (Cloudflare only, no inline Gradio)
import gradio as gr

USE_TUNNEL = True
PASSWORD = "123456"

try: gr.close_all()
except Exception: pass
try: tunnel.terminate()
except NameError: pass
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
subprocess.run("fuser -k 7860/tcp", shell=True, capture_output=True)
time.sleep(2)

def _e(s): return str(s).replace("|", "/").replace("\n", " ")
def fmt(items):
    if not items: return "_No anomalies found._"
    rows = ["| Severity | Type | Finding | Evidence | Found by |", "|---|---|---|---|---|"]
    for a in sorted(items, key=lambda a: (SEV[a["severity"]], a["type"])):
        rows.append(f"| {a['severity']} | {a['type']} | {_e(a['description'])} | {_e(a['evidence'])} | {a['detector']} / {a['source']} |")
    return "\n".join(rows)
def fmt_rej(r):
    items = [f"- [{a['type']}] {_e(a['description'])}  _(rejected: {_e(a.get('reason',''))})_" for a in r["rejected"]]
    items += [f"- [{a['type']}] {_e(a['description'])}  _(evidence not found in the record)_" for a in r["ungrounded"]]
    return "\n".join(items) or "_None._"

def _run_stream(src, k):
    q, res, status = queue.Queue(), {}, []
    def work():
        try: res["r"] = run_record(src, int(k), cb=q.put)
        except Exception: res["e"] = traceback.format_exc()
    th = threading.Thread(target=work, daemon=True); th.start()
    while th.is_alive() or not q.empty():
        try: status.append(q.get(timeout=2))
        except queue.Empty: pass
        yield "", "\n".join(f"… {s}" for s in status[-12:]), "", "", ""
    if "e" in res:
        yield "", "ERROR:\n" + res["e"], "", "", ""; return
    r, rec = res["r"]
    stats = (f"{len(r['final']['verified'])} anomalies (baseline {len(r['baseline']['verified'])}) | rejected {len(r['final']['rejected'])} | "
             f"not grounded {len(r['final']['ungrounded'])} | rules adopted {len(r['adopted'])} | {rec['secs']}s\n"
             "Decision support only. A clinician must verify every finding.")
    edits = "\n".join(f"- [{e['scope']}] {e['edit']}" for e in r["adopted"]) or "(none adopted)"
    yield fmt(r["final"]["verified"]), stats, edits, fmt(r["baseline"]["verified"]), fmt_rej(r["final"])

def ui_split(adm, course, meds, labs, k, deid):
    try:
        if deid: adm, course, meds, labs = map(deidentify, (adm, course, meds, labs))
        yield from _run_stream(build_source(adm, course, meds, labs), k)
    except Exception:
        yield "", "ERROR:\n" + traceback.format_exc(), "", "", ""

def ui_full(text, k, deid):
    try:
        if not (text or "").strip():
            yield "", "Paste the full discharge first.", "", "", ""; return
        yield from _run_stream(deidentify(text) if deid else text, k)
    except Exception:
        yield "", "ERROR:\n" + traceback.format_exc(), "", "", ""

def ui_bench(n, k, seed):
    q, res, lines = queue.Queue(), {}, []
    def work():
        try: benchmark(int(n), int(k), int(seed), cb=q.put); res["ok"] = True
        except Exception: res["e"] = traceback.format_exc()
    th = threading.Thread(target=work, daemon=True); th.start()
    while th.is_alive() or not q.empty():
        try: lines.append(q.get(timeout=2))
        except queue.Empty: pass
        yield "\n".join(lines)
    if "e" in res: lines.append("ERROR:\n" + res["e"])
    yield "\n".join(lines)

def ui_promote():
    n = promote()
    return f"Promoted {len(n)} rule(s) into the base harness." if n else "No rule qualified yet (needs 2+ adoptions and a positive average benchmark gain)."

def outputs():
    return (gr.Markdown(label="Anomalies"), gr.Textbox(lines=4, label="Run stats"), gr.Textbox(lines=4, label="Detector rules adopted by refinement"),
            gr.Markdown(label="Baseline anomalies (before refinement)"), gr.Markdown(label="Rejected / not-grounded candidates (transparency)"))

with gr.Blocks(title="Discharge Anomaly Detector (Harness²-style)") as demo:
    gr.Markdown("## Discharge anomaly detection (local Ollama + recursive harness refinement)\n"
                "**Decision support only. A clinician must verify every finding. Use synthetic or de-identified data on a public link.**")
    with gr.Tab("Full discharge (single text)"):
        full = gr.Textbox(lines=26, label="Full discharge record (one text)",
                          placeholder="Paste the entire discharge as one text: admission, hospital course, vitals, medications, labs, pending results, follow-up...")
        with gr.Row():
            k2 = gr.Slider(0, 3, value=1, step=1, label="Refinement steps k (0 = detect only)")
            d2 = gr.Checkbox(True, label="Scrub identifiers first")
        go2 = gr.Button("Detect anomalies", variant="primary")
        o2 = outputs(); go2.click(ui_full, [full, k2, d2], list(o2))
    with gr.Tab("Split fields"):
        adm = gr.Textbox(lines=4, label="Admission note")
        course = gr.Textbox(lines=5, label="Hospital course / progress notes")
        meds = gr.Textbox(lines=4, label="Medications (home -> discharge)")
        labs = gr.Textbox(lines=4, label="Labs / results / pending")
        with gr.Row():
            k1 = gr.Slider(0, 3, value=1, step=1, label="Refinement steps k (0 = detect only)")
            d1 = gr.Checkbox(True, label="Scrub identifiers first")
        go1 = gr.Button("Detect anomalies", variant="primary")
        o1 = outputs(); go1.click(ui_split, [adm, course, meds, labs, k1, d1], list(o1))
    with gr.Tab("Benchmark (seeded anomalies)"):
        gr.Markdown("Runs synthetic discharges with known injected anomalies and scores precision/recall. k=0 measures the baseline only.")
        with gr.Row():
            bn = gr.Slider(3, 20, value=8, step=1, label="Cases"); bk = gr.Slider(0, 2, value=0, step=1, label="k"); bs = gr.Number(value=0, label="Seed")
        bgo = gr.Button("Run benchmark", variant="primary"); bout = gr.Textbox(lines=22, label="Progress and results")
        bgo.click(ui_bench, [bn, bk, bs], bout)
    with gr.Tab("Insights"):
        rep_md = gr.Markdown()
        gr.Button("Refresh insights").click(insights_report, None, rep_md)
        msg = gr.Textbox(label="")
        gr.Button("Promote proven rules to base harness").click(ui_promote, None, msg)

demo.queue(default_concurrency_limit=2).launch(server_name="0.0.0.0", server_port=7860, share=False, inline=False,
                                               prevent_thread_lock=True, quiet=True, show_error=True, auth=("clinic", PASSWORD))

def start_tunnel(port=7860, timeout=90):
    proc = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://localhost:{port}", "--no-autoupdate"],
                            stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
    lines = queue.Queue()
    threading.Thread(target=lambda: [lines.put(l) for l in proc.stderr], daemon=True).start()
    t0 = time.time()
    while time.time() - t0 < timeout:
        try: line = lines.get(timeout=1)
        except queue.Empty: continue
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
        if m: return proc, m.group(0)
    return proc, None

url = None
if USE_TUNNEL:
    if PASSWORD in ("CHANGE-ME-NOW", "123456", "password"):
        print("WARNING: weak/default password on a public URL. Change PASSWORD and re-run this cell.")
    tunnel, url = start_tunnel()
    print("PUBLIC URL:", url or "not found, check cloudflared output")
    print("Login: clinic /", PASSWORD)

def _ok(u):
    try: return requests.get(u, timeout=15).status_code
    except Exception as e: return f"FAIL ({type(e).__name__})"
print("Local app :", _ok("http://localhost:7860/"))
if url: print("Public URL:", _ok(url))
print("Ollama    :", _ok("http://localhost:11434/"))